# Load Processed Data into Vector Database

This notebook loads output from data prep kit into Milvus

**Step-5 in this workflow**

![](media/rag-overview-2.png)


## Step-1: Configuration

In [1]:
from my_config import MY_CONFIG

## Step-2: Load Parquet Data

Load all  `.parquet` files in the given dir

In [2]:
import pandas as pd
import glob

print ('Loading data from : ', MY_CONFIG.OUTPUT_FOLDER_FINAL)

# Get a list of all Parquet files in the directory
parquet_files = glob.glob(f'{MY_CONFIG.OUTPUT_FOLDER_FINAL}/*.parquet')
print ("Number of parquet files to read : ", len(parquet_files))
print ()

# Create an empty list to store the DataFrames
dfs = []

# Loop through each Parquet file and read it into a DataFrame
for file in parquet_files:
    df = pd.read_parquet(file)
    print (f"Read file: '{file}'.  number of rows = {df.shape[0]}")
    dfs.append(df)

# Concatenate all DataFrames into a single DataFrame
data_df = pd.concat(dfs, ignore_index=True)

print (f"\nTotal number of rows = {data_df.shape[0]}")

Loading data from :  output/output_final
Number of parquet files to read :  2

Read file: 'output/output_final/granite.parquet'.  number of rows = 33
Read file: 'output/output_final/attention.parquet'.  number of rows = 28

Total number of rows = 61


In [3]:

## Shape the data

MY_CONFIG.EMBEDDING_LENGTH =  len(data_df.iloc[0]['embeddings'])
print ('embedding length: ', MY_CONFIG.EMBEDDING_LENGTH)

# rename 'embeddings' columns as 'vector' to match default schema
# if 'vector' not in data_df.columns and 'embeddings' in data_df.columns:
#     data_df = data_df.rename( columns= {'embeddings' : 'vector'})
# if 'text' not in data_df.columns and 'contents' in data_df.columns:
#     data_df = data_df.rename( columns= {'contents' : 'text'})

data_df = data_df.rename( columns= {'embeddings' : 'vector', 'contents' : 'text'})

print (data_df.info())
data_df.head(3)

embedding length:  384
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   filename               61 non-null     object 
 1   num_pages              61 non-null     int64  
 2   num_tables             61 non-null     int64  
 3   num_doc_elements       61 non-null     int64  
 4   document_hash          61 non-null     object 
 5   ext                    61 non-null     object 
 6   hash                   61 non-null     object 
 7   size                   61 non-null     int64  
 8   date_acquired          61 non-null     object 
 9   document_convert_time  61 non-null     float64
 10  source_filename        61 non-null     object 
 11  source_document_id     61 non-null     object 
 12  text                   61 non-null     object 
 13  document_id            61 non-null     object 
 14  vector                 61 non-null   

,filename,num_pages,num_tables,num_doc_elements,document_hash,ext,hash,size,date_acquired,document_convert_time,source_filename,source_document_id,text,document_id,vector
0,granite.pdf,28,17,485,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T09:32:24.162936,59.185795,granite.pdf,adae9d79-6769-4eb7-96c0-b4021394dc25,## Granite Code Models: A Family of Open Found...,4ba39540df65ca93d9dc3026e7ffcea2f949ce9815229c...,"[0.0064111887, -0.030633124, 0.005321988, 0.04..."
1,granite.pdf,28,17,485,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T09:32:24.162936,59.185795,granite.pdf,adae9d79-6769-4eb7-96c0-b4021394dc25,## Abstract\n\nLarge Language Models (LLMs) tr...,fa843bc8b05ba80dd1ee780200b1614af27a2818759fa4...,"[0.026825873, -0.026808968, 0.11900199, 0.0277..."
2,granite.pdf,28,17,485,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T09:32:24.162936,59.185795,granite.pdf,adae9d79-6769-4eb7-96c0-b4021394dc25,## 1 Introduction\n\nOver the last several dec...,d8469bf02cb5f03e01f372eb1c1265acf439855003d1a5...,"[-0.029659774, 0.0077403677, 0.08020694, 0.006..."


## Step-3: Connect to Vector Database

Milvus can be embedded and easy to use.

<span style="color:blue;">Note: If you encounter an error about unable to load database, try this: </span>

- <span style="color:blue;">In **vscode** : **restart the kernel** of previous notebook. This will release the db.lock </span>
- <span style="color:blue;">In **Jupyter**: Do `File --> Close and Shutdown Notebook` of previous notebook. This will release the db.lock</span>
- <span style="color:blue;">Re-run this cell again</span>




In [4]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(MY_CONFIG.DB_URI)

print ("✅ Connected to Milvus instance:", MY_CONFIG.DB_URI)

/Users/shalisha.witherspoonibm.com/Documents/DPK_RAG_UPDATE/data-prep-kit-outer/examples/rag-pdf-1/venv/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


✅ Connected to Milvus instance: ./rag_1_dpk.db


# Step-4: Create A Collection



In [5]:
# if we already have a collection, clear it first
if milvus_client.has_collection(collection_name=MY_CONFIG.COLLECTION_NAME):
    milvus_client.drop_collection(collection_name=MY_CONFIG.COLLECTION_NAME)
    print ('✅ Cleared collection :', MY_CONFIG.COLLECTION_NAME)


milvus_client.create_collection(
    collection_name=MY_CONFIG.COLLECTION_NAME,
    dimension=MY_CONFIG.EMBEDDING_LENGTH,
    metric_type="IP",  # Inner product distance
    consistency_level="Strong",  # Strong consistency level
    auto_id=True
)
print ("✅ Created collection :", MY_CONFIG.COLLECTION_NAME)


✅ Created collection : dpk_papers


## Step-5: Insert Data into Collection

In [6]:
res = milvus_client.insert(collection_name=MY_CONFIG.COLLECTION_NAME, data=data_df.to_dict('records'))

print('inserted # rows', res['insert_count'])

milvus_client.get_collection_stats(MY_CONFIG.COLLECTION_NAME)

inserted # rows 61


{'row_count': 61}

## Step-6: Close DB Connection

Close the connection so the lock files are relinquished and other notebooks can access the db

In [7]:
milvus_client.close()

print ("✅ SUCCESS")


✅ SUCCESS


## Test your data by doing a Vector Search

See notebook [vector_search.ipynb](vector_search.ipynb)